# 06 · LIBERO Shared-Interface Comparison: **Eval-Only** (Liquid vs Pi0-Style)

**Notebook version: 1.12.0-eval**

This notebook is a safe, read-only evaluator for checkpoints produced by the training notebook.

It performs **MuJoCo closed-loop evaluation only** (no offline HDF5 best-of-K metrics and no LIBERO dataset download).

It loads existing checkpoints, runs simulator rollouts, and writes outputs to a separate eval artifacts folder.

It does **not** train and does **not** overwrite training checkpoints.

In [ ]:
# Notebook fingerprint (cache-bust verification)
NOTEBOOK_VERSION = '1.12.0-eval'
NOTEBOOK_GIT_COMMIT = 'eval-only-local'
print(f'NOTEBOOK_VERSION={NOTEBOOK_VERSION}')
print(f'NOTEBOOK_GIT_COMMIT={NOTEBOOK_GIT_COMMIT[:12]}')

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

import yaml

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

# Keep LIBERO config local to the notebook runtime so imports are non-interactive.
LIBERO_CONFIG_HOME = Path('/content/.libero' if IS_COLAB else (Path.cwd() / '.libero'))
os.environ['LIBERO_CONFIG_PATH'] = str(LIBERO_CONFIG_HOME)

# Notebook-local dataset/cache root used to preseed LIBERO's config.
LIBERO_DATA_ROOT = Path('/content/libero_data' if IS_COLAB else (Path.cwd() / 'libero_data'))


def _pip(*pkgs):
    subprocess.check_call(
        [sys.executable, '-m', 'pip', '-q', '--disable-pip-version-check', 'install', *pkgs]
    )


def _preconfigure_libero():
    spec = importlib.util.find_spec('libero.libero')
    if spec is None or spec.origin is None:
        spec = importlib.util.find_spec('libero')
    if spec is None or spec.origin is None:
        raise RuntimeError('LIBERO is not installed, so its config cannot be preseeded.')

    package_root = Path(spec.origin).resolve().parent
    nested_root = package_root / 'libero'
    benchmark_root = nested_root if nested_root.exists() else package_root

    LIBERO_CONFIG_HOME.mkdir(parents=True, exist_ok=True)
    datasets_dir = LIBERO_DATA_ROOT / 'datasets'
    datasets_dir.mkdir(parents=True, exist_ok=True)

    config = {
        'benchmark_root': str(benchmark_root),
        'bddl_files': str(benchmark_root / 'bddl_files'),
        'init_states': str(benchmark_root / 'init_files'),
        'datasets': str(datasets_dir),
        'assets': str(benchmark_root / 'assets'),
    }

    config_path = LIBERO_CONFIG_HOME / 'config.yaml'
    with config_path.open('w') as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print(f'LIBERO config preseeded at {config_path}')
    print(f"LIBERO datasets path: {config['datasets']}")


if IS_COLAB:
    _pip('torch', 'torchvision', 'torchaudio',
         '--index-url', 'https://download.pytorch.org/whl/cu124')
    _pip('transformers', 'accelerate', 'einops', 'tqdm', 'pyyaml', 'ncps',
         'wandb', 'huggingface_hub', 'h5py', 'ftfy', 'regex', 'sentencepiece',
         'imageio', 'imageio-ffmpeg')
    # LIBERO simulator — required for closed-loop rollout evaluation.
    # Needs MuJoCo; on most A100 Colab instances this installs cleanly.
    _pip('libero', 'robomimic', 'gymnasium')
    _preconfigure_libero()
    print('LIBERO simulator installed.')
else:
    # Local dev: install only packages that may be missing
    _pip('transformers', 'accelerate', 'huggingface_hub', 'h5py', 'ncps', 'wandb',
         'tqdm', 'einops', 'ftfy', 'regex', 'sentencepiece', 'imageio', 'imageio-ffmpeg')
    if importlib.util.find_spec('libero') is not None:
        _preconfigure_libero()

print(f'Install complete. IS_COLAB={IS_COLAB}')

In [ ]:
# ================================================================
#  TEST MODE
#
#  TEST = True   →  one full end-to-end pass:
#                   model download, data download, 1 training epoch,
#                   checkpoint save, and W&B upload.
#                   Takes ~5-10 min on A100, ~2 min on CPU (synthetic).
#
#  TEST = False  →  full production run using cfg.epochs / full dataset.
# ================================================================
TEST = False

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
import json
import math
import time
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# IS_COLAB is set by the install cell; define fallback in case that cell was skipped
if 'IS_COLAB' not in globals():
    IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

try:
    from transformers import CLIPModel, CLIPTokenizer
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

try:
    from ncps.torch import CfC
    HAS_NCPS = True
except Exception:
    HAS_NCPS = False

try:
    import wandb
    HAS_WANDB = True
except Exception:
    HAS_WANDB = False

try:
    import h5py
    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False

try:
    from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files
    HAS_HF_HUB = True
except ImportError:
    HAS_HF_HUB = False

LIBERO_IMPORT_STYLE = None
LIBERO_IMPORT_ERROR = None
try:
    import libero  # noqa: F401
    try:
        from libero.envs import OffScreenRenderEnv  # noqa: F401
        LIBERO_IMPORT_STYLE = 'libero.envs'
    except ImportError:
        from libero.libero.envs import OffScreenRenderEnv  # noqa: F401
        LIBERO_IMPORT_STYLE = 'libero.libero.envs'
    HAS_LIBERO = True
except ImportError as e:
    HAS_LIBERO = False
    LIBERO_IMPORT_ERROR = str(e)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(
    f'device={device} | ncps={HAS_NCPS} | transformers={HAS_TRANSFORMERS}'
    f' | wandb={HAS_WANDB} | h5py={HAS_H5PY} | hf_hub={HAS_HF_HUB}'
    f' | libero={HAS_LIBERO}'
)
if LIBERO_IMPORT_STYLE is not None:
    print(f'LIBERO import path: {LIBERO_IMPORT_STYLE}')
elif LIBERO_IMPORT_ERROR is not None:
    print(f'LIBERO import error: {LIBERO_IMPORT_ERROR}')

# --- JEPA-aligned MDN utilities ---
def mdn_nll_loss(logits, mu, log_sigma, target):
    # logits: [B,T,K], mu/log_sigma: [B,T,K,A], target: [B,T,A]
    target = target.unsqueeze(2)  # [B,T,1,A]
    z = (target - mu) * torch.exp(-log_sigma)
    log_prob = -0.5 * (z * z + 2.0 * log_sigma + math.log(2.0 * math.pi)).sum(dim=-1)  # [B,T,K]
    log_mix = F.log_softmax(logits, dim=-1) + log_prob
    nll = -torch.logsumexp(log_mix, dim=-1)  # [B,T]
    return nll.mean()

@torch.no_grad()
def sample_mdn_actions(logits, mu, log_sigma, K=10):
    # Returns [B,K,T,A]
    B, T, M = logits.shape
    A = mu.shape[-1]
    probs = F.softmax(logits, dim=-1)
    comp = torch.distributions.Categorical(probs=probs)
    idx = comp.sample((K,)).permute(1, 0, 2)  # [B,K,T]
    eps = torch.randn(B, K, T, A, device=mu.device)

    idx_exp = idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, -1, 1, A)
    mu_k = mu.unsqueeze(1).expand(B, K, T, M, A).gather(3, idx_exp).squeeze(3)
    ls_k = log_sigma.unsqueeze(1).expand(B, K, T, M, A).gather(3, idx_exp).squeeze(3)
    return mu_k + torch.exp(ls_k) * eps

@torch.no_grad()
def mdn_mean_actions(logits, mu):
    w = F.softmax(logits, dim=-1)
    return (w.unsqueeze(-1) * mu).sum(dim=2)  # [B,T,A]

@torch.no_grad()
def compute_jerk(trajs):
    if trajs.shape[-2] < 3:
        return torch.zeros(trajs.shape[:-2], device=trajs.device)
    jerk = trajs[..., 2:, :] - 2.0 * trajs[..., 1:-1, :] + trajs[..., :-2, :]
    return (jerk ** 2).mean(dim=(-1, -2))

@torch.no_grad()
def compute_sample_metrics(samples, target):
    # samples: [B,K,T,A], target: [B,T,A]
    per_step_mse = ((samples - target.unsqueeze(1)) ** 2).mean(dim=-1)  # [B,K,T]
    traj_mse = per_step_mse.mean(dim=-1)  # [B,K]
    best_idx = traj_mse.argmin(dim=1)
    batch_idx = torch.arange(samples.shape[0], device=samples.device)
    best_step_mse = per_step_mse[batch_idx, best_idx]

    if samples.shape[1] > 1:
        diffs = samples.unsqueeze(2) - samples.unsqueeze(1)
        pairwise_rms = torch.sqrt((diffs ** 2).mean(dim=(-1, -2)) + 1e-12)
        tri = torch.triu_indices(samples.shape[1], samples.shape[1], offset=1, device=samples.device)
        diversity = pairwise_rms[:, tri[0], tri[1]].mean(dim=1).mean().item()
    else:
        diversity = 0.0

    return {
        'sample_mean_mse': traj_mse.mean().item(),
        'best_of_k_mse': traj_mse.min(dim=1).values.mean().item(),
        'diversity_l2': diversity,
        'smoothness': compute_jerk(samples).mean().item(),
        'per_horizon_best_of_k_mse': best_step_mse.mean(dim=0).detach().cpu().tolist(),
    }

device = cpu | ncps = False | transformers = True


In [ ]:
@dataclass
class Config:
    # Shared task setup
    obs_horizon: int = 2
    pred_horizon: int = 16
    state_dim: int = 14
    action_dim: int = 7
    vocab_size: int = 49408  # CLIP vocab size
    text_len: int = 32

    # Shared backbone (must match training architecture)
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 8
    dropout: float = 0.1

    # Head sizes + size policy (must match training architecture)
    liquid_hidden: int = 1536
    pi0_hidden: int = 1792
    num_mixtures: int = 8
    mdn_head_depth: int = 2
    mdn_head_dropout: float = 0.10
    log_sigma_min: float = -5.0
    enforce_liquid_half_size: bool = True
    target_liquid_to_pi0_ratio: float = 0.50
    liquid_d_model: int = 768
    liquid_n_heads: int = 12
    liquid_n_layers: int = 7

    # Eval-only loader settings
    batch_size: int = 48
    loader_num_workers: int = 0

    # Encoder strategy (must match training)
    use_pretrained_encoders: bool = True
    pretrained_name: str = 'openai/clip-vit-large-patch14'
    freeze_encoders: bool = True
    unfreeze_clip_vision_last_n: int = 0
    unfreeze_clip_text_last_n: int = 0
    unfreeze_clip_projections: bool = False
    image_size: int = 224

    # MuJoCo-only eval protocol
    eval_mode: str = 'mujoco_only'
    n_eval_episodes: int = 10
    eval_horizon: int = 300
    eval_k_samples: int = 10
    closed_loop_eval_tasks: int = 10
    chunk_exec_len: int = 8
    closed_loop_action_selection: str = 'mode'

    # Dataset suite name is still used to select BDDL subfolder under bddl_files/
    libero_suite: str = 'LIBERO_90'

    # Storage
    use_drive: bool = True
    drive_mount_point: str = '/content/drive'
    drive_root: str = '/content/drive/MyDrive/liquidnets_runs'
    local_root: str = '/content/liquidnets_runs'

    # Read-only checkpoint source + isolated eval output
    source_run_name: str = 'libero90_liquid_vs_pi0_scaled'
    eval_run_name: str = 'libero90_liquid_vs_pi0_scaled_eval'

    # Tracking (off by default for safety)
    enable_wandb: bool = False
    wandb_project: str = 'liquidnets-libero'
    wandb_entity: str = ''

cfg = Config()

# Apply TEST-mode overrides
_test = globals().get('TEST', False)
if _test:
    cfg.batch_size = 8
    cfg.n_eval_episodes = 1
    cfg.closed_loop_eval_tasks = 1

print(f'TEST={_test} | eval_mode={cfg.eval_mode} | batch_size={cfg.batch_size} | workers={cfg.loader_num_workers}')
print(f'source_run_name={cfg.source_run_name} | eval_run_name={cfg.eval_run_name}')
print(f'd_model={cfg.d_model} | n_heads={cfg.n_heads} | num_mixtures={cfg.num_mixtures}')
print(f'pretrained_name={cfg.pretrained_name}')
print(f'mdn_head_depth={cfg.mdn_head_depth} | mdn_head_dropout={cfg.mdn_head_dropout} | log_sigma_min={cfg.log_sigma_min}')
print(f'chunk_exec_len={cfg.chunk_exec_len} | action_selection={cfg.closed_loop_action_selection}')
print('Mode: EVAL-ONLY + MuJoCo-ONLY (no training, no HDF5 dataset eval)')

In [ ]:
# Runtime paths, Drive mount (Colab), and optional W&B init
IS_COLAB = 'COLAB_GPU' in os.environ

if cfg.use_drive and IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount(cfg.drive_mount_point, force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive mount failed: {e}. Falling back to local path.')

root_path = Path(cfg.drive_root if (cfg.use_drive and Path(cfg.drive_mount_point).exists()) else cfg.local_root)

# ---- Read-only checkpoint source ----
source_run_path = root_path / cfg.source_run_name
source_ckpt_path = source_run_path / 'checkpoints'
latest_ckpt = source_ckpt_path / 'latest.pt'
best_liq_ckpt = source_ckpt_path / 'best_liquid.pt'
best_pi0_ckpt = source_ckpt_path / 'best_pi0.pt'

# ---- Isolated eval output path ----
eval_run_path = root_path / cfg.eval_run_name
artifact_path = eval_run_path / 'artifacts'
artifact_path.mkdir(parents=True, exist_ok=True)
history_json = artifact_path / 'history_eval.json'
results_json = artifact_path / 'results_eval.json'

print('Checkpoint source:', source_ckpt_path)
print('Eval output path :', artifact_path)
print('Latest checkpoint:', latest_ckpt)
print('Best liquid      :', best_liq_ckpt)
print('Best pi0         :', best_pi0_ckpt)

if not (best_liq_ckpt.exists() and best_pi0_ckpt.exists()):
    raise FileNotFoundError(
        'Missing best checkpoints in source run. '\
        f'Expected: {best_liq_ckpt} and {best_pi0_ckpt}'
    )

# Optional W&B (disabled by default). Uses separate run name.
wandb_run = None
if cfg.enable_wandb and HAS_WANDB:
    wb_kwargs = {
        'project': cfg.wandb_project,
        'name': cfg.eval_run_name,
        'config': cfg.__dict__,
        'reinit': True,
    }
    if cfg.wandb_entity:
        wb_kwargs['entity'] = cfg.wandb_entity
    wandb_run = wandb.init(**wb_kwargs)
    print('W&B eval run initialized:', wandb_run.name)
elif cfg.enable_wandb:
    print('[WARN] wandb not installed in kernel; continuing without online logging.')

print('[SAFE MODE] This notebook does not write training checkpoints.')

In [ ]:
# ---------------------------------------------------------------
# Optional: redirect HuggingFace cache to Drive so models survive
# Colab session restarts without re-downloading.
# ---------------------------------------------------------------
if cfg.use_drive and IS_COLAB and Path(cfg.drive_mount_point).exists():
    hf_cache = root_path / 'hf_cache'
    hf_cache.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(hf_cache)
    print(f'HF_HOME → {hf_cache}')

# ---------------------------------------------------------------
# CLIP model pre-download
# ---------------------------------------------------------------
clip_tokenizer = None
if cfg.use_pretrained_encoders:
    if HAS_HF_HUB:
        print(f'Pre-caching {cfg.pretrained_name} ...')
        snapshot_download(
            repo_id=cfg.pretrained_name,
            repo_type='model',
            ignore_patterns=['*.msgpack', '*.h5', 'flax_*', 'tf_*', 'rust_model.ot'],
        )
        print('  model weights cached.')
    else:
        print('[INFO] huggingface_hub not available; transformers will auto-download on first use.')

    if HAS_TRANSFORMERS:
        try:
            clip_tokenizer = CLIPTokenizer.from_pretrained(cfg.pretrained_name)
            print(f'  CLIP tokenizer loaded ({cfg.pretrained_name}).')
        except Exception as e:
            print(f'  [WARN] Tokenizer load failed: {e}. Task descriptions will use hash-based token fallback.')

# ---------------------------------------------------------------
# MuJoCo-only evaluation mode: no HDF5 demo download.
# We only need LIBERO simulator + BDDL tasks from the installed package.
# ---------------------------------------------------------------
LIBERO_LOCAL_PATH = root_path / 'libero_data'
LIBERO_LOCAL_PATH.mkdir(parents=True, exist_ok=True)

LIBERO_SUITE_ALIASES = {
    'LIBERO_SPATIAL': 'libero_spatial',
    'LIBERO_OBJECT': 'libero_object',
    'LIBERO_GOAL': 'libero_goal',
    'LIBERO_10': 'libero_10',
    'LIBERO_90': 'libero_90',
    'LIBERO_LONG': 'libero_90',
}

def get_libero_suite_dir_name(suite_name: str) -> str:
    suite_name = str(suite_name).strip()
    return LIBERO_SUITE_ALIASES.get(suite_name, suite_name.lower())

cfg.libero_suite = get_libero_suite_dir_name(cfg.libero_suite)
print('[INFO] MuJoCo-only mode: skipping LIBERO HDF5 dataset download and offline metrics.')
print(f'libero_suite={cfg.libero_suite} | LIBERO_LOCAL_PATH={LIBERO_LOCAL_PATH}')

In [ ]:
class Batch(dict):
    """Keys used by both models: image,state,text_ids,text_mask,action"""


class TinyVision(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Linear(128, d)

    def forward(self, x):
        B, T = x.shape[:2]
        h = self.net(x.reshape(B * T, *x.shape[2:])).flatten(1)
        return self.proj(h).view(B, T, -1)


class TinyText(nn.Module):
    def __init__(self, vocab_size, d):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d)

    def forward(self, text_ids, text_mask):
        h = self.tok(text_ids)
        m = text_mask.float().unsqueeze(-1)
        return (h * m).sum(1) / m.sum(1).clamp_min(1.0)


class PretrainedClipEncoders(nn.Module):
    """Pretrained vision+text interface that can be recycled across Liquid/Pi0 heads."""
    def __init__(self, cfg):
        super().__init__()
        self.enabled = bool(cfg.use_pretrained_encoders and HAS_TRANSFORMERS)
        self.image_size = int(cfg.image_size)
        self.vision_out_dim = cfg.d_model
        self.text_out_dim = cfg.d_model

        if self.enabled:
            try:
                clip = CLIPModel.from_pretrained(cfg.pretrained_name)
                self.vision = clip.vision_model
                self.text = clip.text_model
                self.visual_projection = clip.visual_projection
                self.text_projection = clip.text_projection
                self.vision_out_dim = int(self.visual_projection.out_features)
                self.text_out_dim = int(self.text_projection.out_features)
            except Exception as e:
                print(f'[WARN] Failed to load pretrained encoders: {e}. Falling back to tiny encoders.')
                self.enabled = False

        if not self.enabled:
            self.vision = TinyVision(cfg.d_model)
            self.text = TinyText(cfg.vocab_size, cfg.d_model)
            self.visual_projection = nn.Identity()
            self.text_projection = nn.Identity()

    def _preprocess_image(self, x):
        # x: [B,T,3,H,W], expected ~[0,1]
        B, T = x.shape[:2]
        x = x.reshape(B * T, *x.shape[2:])
        x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        # CLIP normalization
        mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1, 3, 1, 1)
        std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std
        return x, B, T

    def encode_vision(self, image):
        if self.enabled:
            x, B, T = self._preprocess_image(image)
            out = self.vision(pixel_values=x)
            cls = out.last_hidden_state[:, 0]
            feat = self.visual_projection(cls)
            return feat.view(B, T, -1)
        return self.vision(image)

    def encode_text(self, text_ids, text_mask):
        if self.enabled:
            out = self.text(input_ids=text_ids, attention_mask=text_mask)
            cls = out.last_hidden_state[:, 0]
            return self.text_projection(cls)
        return self.text(text_ids, text_mask)


class SharedBackbone(nn.Module):
    def __init__(self, cfg, shared_encoders=None):
        super().__init__()
        self.cfg = cfg
        self.enc = shared_encoders if shared_encoders is not None else PretrainedClipEncoders(cfg)

        if cfg.freeze_encoders:
            for p in self.enc.parameters():
                p.requires_grad = False

            # Optional partial unfreeze for CLIP encoders
            v_last = int(getattr(cfg, 'unfreeze_clip_vision_last_n', 0))
            t_last = int(getattr(cfg, 'unfreeze_clip_text_last_n', 0))
            unfreeze_proj = bool(getattr(cfg, 'unfreeze_clip_projections', True))

            if getattr(self.enc, 'enabled', False):
                if v_last > 0 and hasattr(self.enc.vision, 'encoder') and hasattr(self.enc.vision.encoder, 'layers'):
                    v_layers = self.enc.vision.encoder.layers
                    for lyr in list(v_layers)[-v_last:]:
                        for p in lyr.parameters():
                            p.requires_grad = True

                if t_last > 0 and hasattr(self.enc.text, 'encoder') and hasattr(self.enc.text.encoder, 'layers'):
                    t_layers = self.enc.text.encoder.layers
                    for lyr in list(t_layers)[-t_last:]:
                        for p in lyr.parameters():
                            p.requires_grad = True

                if unfreeze_proj:
                    for p in self.enc.visual_projection.parameters():
                        p.requires_grad = True
                    for p in self.enc.text_projection.parameters():
                        p.requires_grad = True

            if not any(p.requires_grad for p in self.enc.parameters()):
                self.enc.eval()
            else:
                _n_trainable = sum(int(p.numel()) for p in self.enc.parameters() if p.requires_grad)
                print(f'[INFO] Partial encoder unfreeze active: trainable encoder params={_n_trainable:,}')

        self.vis_proj = nn.Linear(self.enc.vision_out_dim, cfg.d_model)
        self.state_proj = nn.Linear(cfg.state_dim, cfg.d_model)
        self.text_proj = nn.Linear(self.enc.text_out_dim, cfg.d_model)

        layer = nn.TransformerEncoderLayer(
            cfg.d_model,
            cfg.n_heads,
            4 * cfg.d_model,
            cfg.dropout,
            batch_first=True,
        )
        self.temporal = nn.TransformerEncoder(layer, cfg.n_layers)

    def forward(self, batch):
        with torch.set_grad_enabled(any(p.requires_grad for p in self.enc.parameters())):
            v = self.enc.encode_vision(batch['image'])
            t = self.enc.encode_text(batch['text_ids'], batch['text_mask'])

        v = self.vis_proj(v)
        s = self.state_proj(batch['state'])
        t = self.text_proj(t).unsqueeze(1).expand(-1, v.size(1), -1)
        return self.temporal(v + s + t)

In [ ]:
class MdnHead(nn.Module):
    """Mixture Density Network head with configurable hidden depth and regularisation.

    hidden_depth > 0 inserts that many SiLU hidden layers before the output,
    giving the head more representational capacity.

    dropout: applied between the last hidden layer and the final linear to
    regularise the action distribution.  Prevents the MDN from learning
    overly sharp (memorising) variance estimates on the train set.

    log_sigma_min: tighter lower clamp on log-variance (default -5.0, was -7.0).
    Prevents collapsed Gaussians that drive train NLL down without generalising.
    """
    def __init__(self, in_dim, action_dim, num_mixtures, hidden_depth=2,
                 dropout=0.0, log_sigma_min=-5.0):
        super().__init__()
        self.action_dim = action_dim
        self.num_mixtures = num_mixtures
        self.log_sigma_min = float(log_sigma_min)
        out_dim = num_mixtures * (1 + 2 * action_dim)
        layers = []
        cur_dim = in_dim
        for _ in range(hidden_depth):
            nxt_dim = max(out_dim * 4, cur_dim // 2)
            layers += [nn.Linear(cur_dim, nxt_dim), nn.SiLU()]
            cur_dim = nxt_dim
        if dropout > 0.0:
            layers.append(nn.Dropout(p=float(dropout)))
        layers.append(nn.Linear(cur_dim, out_dim))
        self.proj = nn.Sequential(*layers)

    def forward(self, h):
        # h: [B,T,H]
        B, T, _ = h.shape
        raw = self.proj(h).view(B, T, self.num_mixtures, 1 + 2 * self.action_dim)
        logits = raw[..., 0]
        mu = raw[..., 1:1 + self.action_dim]
        log_sigma = raw[..., 1 + self.action_dim:]
        log_sigma = torch.clamp(log_sigma, min=self.log_sigma_min, max=2.0)
        return logits, mu, log_sigma


class BasePolicy(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        self.cfg = cfg

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        raise NotImplementedError

    def forward(self, batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True):
        z = self.backbone(batch)
        logits, mu, log_sigma = self.predict_mdn(
            z,
            batch,
            action_target=action_target,
            teacher_forcing_ratio=teacher_forcing_ratio,
        )
        pred_mean = mdn_mean_actions(logits, mu)
        out = {
            'pred_actions': pred_mean,
            'aux': {
                'logits': logits,
                'mu': mu,
                'log_sigma': log_sigma,
            }
        }
        return out

    def loss(self, out, batch):
        logits = out['aux']['logits']
        mu = out['aux']['mu']
        log_sigma = out['aux']['log_sigma']
        nll = mdn_nll_loss(logits, mu, log_sigma, batch['action'])
        mse = F.mse_loss(out['pred_actions'], batch['action'])
        return {'loss': nll, 'nll': nll.detach(), 'mse': mse.detach()}


def _mdn_kwargs(cfg):
    """Shared MdnHead kwargs derived from cfg fields, with safe defaults."""
    return dict(
        hidden_depth=int(getattr(cfg, 'mdn_head_depth', 2)),
        dropout=float(getattr(cfg, 'mdn_head_dropout', 0.0)),
        log_sigma_min=float(getattr(cfg, 'log_sigma_min', -5.0)),
    )


class LiquidPolicy(BasePolicy):
    def __init__(self, backbone, cfg):
        super().__init__(backbone, cfg)
        if HAS_NCPS:
            self.core = CfC(cfg.d_model, cfg.liquid_hidden, batch_first=True)
        else:
            self.core = nn.GRU(cfg.d_model, cfg.liquid_hidden, num_layers=2, batch_first=True)

        self.action_embed = nn.Linear(cfg.action_dim, cfg.d_model)
        self.decoder = nn.GRUCell(cfg.d_model, cfg.liquid_hidden)
        self.mdn = MdnHead(cfg.liquid_hidden, cfg.action_dim, cfg.num_mixtures,
                           **_mdn_kwargs(cfg))

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        B, _, _ = z.shape
        T = self.cfg.pred_horizon

        h_enc, _ = self.core(z)
        h = h_enc[:, -1]  # [B,H]

        prev_action = torch.zeros(B, self.cfg.action_dim, device=z.device)
        hidden_seq = []

        for t in range(T):
            use_teacher = (
                (action_target is not None) and
                (torch.rand(1, device=z.device).item() < float(teacher_forcing_ratio))
            )
            inp_action = action_target[:, t] if use_teacher else prev_action
            inp = self.action_embed(inp_action)
            h = self.decoder(inp, h)
            hidden_seq.append(h.unsqueeze(1))

            # rollout with mean prediction
            logits_t, mu_t, _ = self.mdn(h.unsqueeze(1))
            prev_action = mdn_mean_actions(logits_t, mu_t).squeeze(1)

        hidden_seq = torch.cat(hidden_seq, dim=1)  # [B,T,H]
        return self.mdn(hidden_seq)


class Pi0StylePolicy(BasePolicy):
    def __init__(self, backbone, cfg):
        super().__init__(backbone, cfg)
        self.query = nn.Parameter(torch.randn(1, cfg.pred_horizon, cfg.d_model) * 0.02)
        layer = nn.TransformerDecoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=cfg.dropout,
            batch_first=True,
        )
        self.dec = nn.TransformerDecoder(layer, cfg.n_layers)
        self.proj = nn.Linear(cfg.d_model, cfg.pi0_hidden)
        self.mdn = MdnHead(cfg.pi0_hidden, cfg.action_dim, cfg.num_mixtures,
                           **_mdn_kwargs(cfg))

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        q = self.query.expand(z.size(0), -1, -1)
        y = self.dec(q, z)
        h = self.proj(y)
        return self.mdn(h)

In [ ]:
def collate(items):
    out = Batch()
    for k in items[0].keys():
        out[k] = torch.stack([x[k] for x in items], 0)
    return out


def to_device(batch, d):
    return Batch({k: (v.to(d) if torch.is_tensor(v) else v) for k, v in batch.items()})


def make_tokenizer_fn(cfg, clip_tokenizer=None, task_name: str = ''):
    """Return a callable(obs_dict) -> Batch[B=1] for closed-loop rollout."""
    def _tokenize_obs(obs: dict) -> 'Batch':
        img_np = obs.get('agentview_image', np.zeros((cfg.image_size, cfg.image_size, 3), dtype=np.uint8))
        img = torch.from_numpy(img_np.astype(np.float32) / 255.0).permute(2, 0, 1)
        image = img.unsqueeze(0).expand(cfg.obs_horizon, -1, -1, -1).clone()

        eef_pos  = np.asarray(obs.get('robot0_eef_pos',      [0.0] * 3), dtype=np.float32)[:3]
        eef_quat = np.asarray(obs.get('robot0_eef_quat',     [0.0] * 4), dtype=np.float32)[:4]
        gripper  = np.asarray(obs.get('robot0_gripper_qpos', [0.0] * 2), dtype=np.float32)[:2]
        joint    = np.asarray(obs.get('robot0_joint_pos',    [0.0] * 7), dtype=np.float32)[:5]
        state_np = np.concatenate([eef_pos, eef_quat, gripper, joint])
        state = torch.from_numpy(state_np).unsqueeze(0).expand(cfg.obs_horizon, -1).clone()

        _name = obs.get('_task_name', task_name)
        if clip_tokenizer is not None and _name:
            enc = clip_tokenizer(
                _name, return_tensors='pt', padding='max_length',
                max_length=cfg.text_len, truncation=True,
            )
            text_ids = enc['input_ids'].squeeze(0)
            text_mask = enc['attention_mask'].squeeze(0)
        else:
            text_ids = torch.zeros(cfg.text_len, dtype=torch.long)
            text_mask = torch.zeros(cfg.text_len, dtype=torch.long)

        action = torch.zeros(cfg.pred_horizon, cfg.action_dim)

        return Batch(
            image=image.unsqueeze(0),
            state=state.unsqueeze(0),
            text_ids=text_ids.unsqueeze(0),
            text_mask=text_mask.unsqueeze(0),
            action=action.unsqueeze(0),
        )

    return _tokenize_obs


print('MuJoCo helpers ready.')

In [ ]:
# MuJoCo closed-loop env compatibility helpers

def _get_libero_bddl_root():
    """Resolve LIBERO bddl_files root from config/env when available."""
    cfg_path = os.environ.get('LIBERO_CONFIG_PATH', '')
    if cfg_path:
        cfg_file = Path(cfg_path) / 'config.yaml'
        if cfg_file.exists():
            try:
                with cfg_file.open('r') as f:
                    cfg_data = yaml.safe_load(f) or {}
                root = cfg_data.get('bddl_files', None)
                if root:
                    root_p = Path(root)
                    if root_p.exists():
                        return root_p
            except Exception:
                pass

    try:
        spec = importlib.util.find_spec('libero.libero')
        if spec is None or spec.origin is None:
            spec = importlib.util.find_spec('libero')
        if spec and spec.origin:
            package_root = Path(spec.origin).resolve().parent
            nested_root = package_root / 'libero'
            benchmark_root = nested_root if nested_root.exists() else package_root
            bddl_root = benchmark_root / 'bddl_files'
            if bddl_root.exists():
                return bddl_root
    except Exception:
        pass

    return None


def _resolve_bddl_path(bddl_file_name):
    if not bddl_file_name:
        return None

    bddl = str(bddl_file_name)
    if not bddl.endswith('.bddl'):
        bddl = f"{bddl}.bddl"

    bddl_p = Path(bddl)
    if bddl_p.is_absolute() and bddl_p.exists():
        return str(bddl_p)

    bddl_root = _get_libero_bddl_root()
    if bddl_root is None:
        return bddl

    direct = bddl_root / bddl
    if direct.exists():
        return str(direct)

    target_name = Path(bddl).name
    matches = sorted(bddl_root.rglob(target_name))
    if matches:
        return str(matches[0])

    return bddl


def make_libero_env_fn(task_name: str, image_size: int = 224, bddl_file_name: str = None):
    """Return a callable(seed) -> gym env for a single LIBERO task."""
    def _make(seed: int = 0):
        if not HAS_LIBERO:
            raise RuntimeError(
                'LIBERO simulator not installed. '
                'Re-run the install cell — libero/robomimic/gymnasium must install cleanly.'
            )

        cam_kwargs = {
            'camera_heights': image_size,
            'camera_widths': image_size,
            'camera_names': 'agentview',
        }

        bddl_resolved = _resolve_bddl_path(bddl_file_name)

        attempt_kwargs = []
        if task_name and bddl_resolved:
            attempt_kwargs.append({'task_name': task_name, 'bddl_file_name': bddl_resolved})
        if bddl_resolved:
            attempt_kwargs.append({'bddl_file_name': bddl_resolved})
        if task_name:
            attempt_kwargs.append({'task_name': task_name})

        env = None
        last_error = None
        for kw in attempt_kwargs:
            try:
                env = OffScreenRenderEnv(**kw, **cam_kwargs)
                break
            except Exception as e:
                last_error = e

        if env is None:
            tried = ', '.join([str(x) for x in attempt_kwargs])
            raise RuntimeError(
                f'Failed to construct OffScreenRenderEnv. Tried: {tried}. Last error: {last_error}'
            )

        if hasattr(env, 'seed'):
            env.seed(seed)
        return env

    return _make


print('MuJoCo env helpers ready.')

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


@torch.no_grad()
def latency_ms(model, batch, steps=100):
    import time
    batch = to_device(batch, device)
    model.eval()
    for _ in range(20):
        _ = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(steps):
        _ = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return 1000.0 * (time.perf_counter() - t0) / steps

In [ ]:
# Config + TEST status summary
_test = globals().get('TEST', False)
print(f'TEST             = {_test}')
print(f'eval_mode        = {cfg.eval_mode}')
print(f'pretrained_enc   = {cfg.use_pretrained_encoders}  ({cfg.pretrained_name})')
print(f'freeze_encoders  = {cfg.freeze_encoders}')
print(f'enable_wandb     = {cfg.enable_wandb}')
print(f'run_name         = {cfg.eval_run_name}')
print(f'libero_suite     = {cfg.libero_suite}')
print(f'clip_tokenizer   = {"loaded" if globals().get("clip_tokenizer") else "fallback hash"}')
print(f'checkpoint src   = {globals().get("source_ckpt_path", "not set (setup cell not run)")}')
print(f'── closed-loop eval ──────────────────────────────────────')
print(f'HAS_LIBERO       = {HAS_LIBERO}')
print(f'n_eval_episodes  = {cfg.n_eval_episodes}  (TEST overrides to 1)')
print(f'eval_horizon     = {cfg.eval_horizon}  steps / episode')
print(f'eval_k_samples   = {cfg.eval_k_samples}  (best-of-K for action selection)')

Smoke test mode: False
Current config: {'use_pretrained_encoders': True, 'epochs': 3}


In [ ]:
import json
from copy import deepcopy

# MuJoCo-only evaluation: no HDF5 loaders, no offline best-of-K metrics
run_name = cfg.eval_run_name

# ---- Models: same architecture as training ----
liquid_cfg = deepcopy(cfg)
pi0_cfg = deepcopy(cfg)

if bool(getattr(cfg, 'enforce_liquid_half_size', True)):
    liquid_cfg.d_model = int(getattr(cfg, 'liquid_d_model', liquid_cfg.d_model))
    liquid_cfg.n_heads = int(getattr(cfg, 'liquid_n_heads', liquid_cfg.n_heads))
    liquid_cfg.n_layers = int(getattr(cfg, 'liquid_n_layers', liquid_cfg.n_layers))
    liquid_cfg.liquid_hidden = int(getattr(cfg, 'liquid_hidden', liquid_cfg.liquid_hidden))

# Make sure liquid attention heads divide d_model
liquid_cfg.n_heads = max(1, min(liquid_cfg.n_heads, liquid_cfg.d_model))
while liquid_cfg.n_heads > 1 and (liquid_cfg.d_model % liquid_cfg.n_heads != 0):
    liquid_cfg.n_heads -= 1

shared_encoders = PretrainedClipEncoders(cfg)
backbone_liquid = SharedBackbone(liquid_cfg, shared_encoders=shared_encoders)
backbone_pi0 = SharedBackbone(pi0_cfg, shared_encoders=shared_encoders)

liquid = LiquidPolicy(backbone_liquid, liquid_cfg).to(device)
pi0 = Pi0StylePolicy(backbone_pi0, pi0_cfg).to(device)

print('Pretrained encoders enabled:', cfg.use_pretrained_encoders and HAS_TRANSFORMERS)
print('Freeze encoders:', cfg.freeze_encoders)
print('eval_mode: mujoco_only')
print(f'Liquid arch: d_model={liquid_cfg.d_model}, n_layers={liquid_cfg.n_layers}, n_heads={liquid_cfg.n_heads}, hidden={liquid_cfg.liquid_hidden}')
print(f'Pi0 arch   : d_model={pi0_cfg.d_model}, n_layers={pi0_cfg.n_layers}, n_heads={pi0_cfg.n_heads}, hidden={pi0_cfg.pi0_hidden}')

# ---- Parameter counts (exclude shared encoder so ratio reflects model-specific capacity) ----
_enc_trainable = sum(p.numel() for p in shared_encoders.parameters() if p.requires_grad)
_liq_total = count_params(liquid)
_pi0_total = count_params(pi0)
_liq_excl = _liq_total - _enc_trainable
_pi0_excl = _pi0_total - _enc_trainable
print(f'Shared encoder trainable params: {_enc_trainable:,}')
print(f'Liquid params: {_liq_total:,}  (model-specific: {_liq_excl:,})')
print(f'Pi0-style params: {_pi0_total:,}  (model-specific: {_pi0_excl:,})')
print(f'Liquid / Pi0 ratio (model-specific): {_liq_excl / max(1, _pi0_excl):.3f}x')

# ---- Load checkpoint weights (read-only) ----
def _load_model_weights(model, ckpt_path, model_key_fallback=None):
    payload = torch.load(ckpt_path, map_location=device)
    if isinstance(payload, dict) and 'model' in payload:
        state = payload['model']
    elif model_key_fallback is not None and isinstance(payload, dict) and model_key_fallback in payload:
        state = payload[model_key_fallback]
    else:
        raise KeyError(f'Could not find model weights in {ckpt_path}')
    model.load_state_dict(state)

if best_liq_ckpt.exists():
    _load_model_weights(liquid, best_liq_ckpt)
    print(f'Loaded Liquid weights from best checkpoint: {best_liq_ckpt}')
else:
    _load_model_weights(liquid, latest_ckpt, model_key_fallback='liquid_model')
    print(f'[WARN] best_liquid.pt missing; loaded from latest: {latest_ckpt}')

if best_pi0_ckpt.exists():
    _load_model_weights(pi0, best_pi0_ckpt)
    print(f'Loaded Pi0 weights from best checkpoint: {best_pi0_ckpt}')
else:
    _load_model_weights(pi0, latest_ckpt, model_key_fallback='pi0_model')
    print(f'[WARN] best_pi0.pt missing; loaded from latest: {latest_ckpt}')

# ---- Latency batch from synthetic observation ----
_tok = make_tokenizer_fn(cfg, clip_tokenizer=globals().get('clip_tokenizer', None), task_name='')
_dummy_obs = {
    'agentview_image': np.zeros((cfg.image_size, cfg.image_size, 3), dtype=np.uint8),
    'robot0_eef_pos': np.zeros(3, dtype=np.float32),
    'robot0_eef_quat': np.zeros(4, dtype=np.float32),
    'robot0_gripper_qpos': np.zeros(2, dtype=np.float32),
    'robot0_joint_pos': np.zeros(7, dtype=np.float32),
}
_b = _tok(_dummy_obs)
liq_latency = latency_ms(liquid, _b)
pi0_latency = latency_ms(pi0, _b)
print(f'\nLatency (ms/batch)  Liquid: {liq_latency:.3f}  Pi0-style: {pi0_latency:.3f}')

# ---- Closed-loop evaluation (MuJoCo via LIBERO simulator) ----
_closed_loop_results = {}
if HAS_LIBERO:
    print('\n── Closed-loop rollout evaluation ──────────────────────────────────')

    _bddl_root = _get_libero_bddl_root()
    if _bddl_root is None:
        raise RuntimeError('Could not resolve LIBERO bddl_files root for mujoco_only evaluation.')

    _suite_bddl_dir = _bddl_root / str(cfg.libero_suite)
    _bddl_files = sorted(_suite_bddl_dir.glob('*.bddl')) if _suite_bddl_dir.exists() else sorted(_bddl_root.rglob('*.bddl'))
    _bddl_files = _bddl_files[:max(0, int(getattr(cfg, 'closed_loop_eval_tasks', 10)))]

    _eval_specs = []
    for _bp in _bddl_files:
        _task_label = _bp.stem.replace('_', ' ')
        _eval_specs.append({
            'task_label': _task_label,
            'task_name': _task_label,
            'bddl_file_name': str(_bp),
        })

    if _eval_specs:
        print(f'  Evaluating {len(_eval_specs)} task(s) with {cfg.n_eval_episodes} episode(s)/task/model')
        _closed_loop_video_dir = artifact_path / 'closed_loop_videos'
        _closed_loop_video_dir.mkdir(parents=True, exist_ok=True)

        for _mname, _model in [('liquid', liquid), ('pi0', pi0)]:
            _task_runs = []
            for _ti, _spec in enumerate(_eval_specs):
                _task_label = _spec['task_label']
                _make_env = make_libero_env_fn(
                    _spec.get('task_name'),
                    image_size=cfg.image_size,
                    bddl_file_name=_spec.get('bddl_file_name'),
                )
                _tok_fn = make_tokenizer_fn(cfg, clip_tokenizer=globals().get('clip_tokenizer', None), task_name=_task_label)

                print(f'  Evaluating {_mname}: task={_ti+1}/{len(_eval_specs)} "{_task_label}"')
                try:
                    _cl = evaluate_closed_loop(
                        _model, _make_env, _tok_fn,
                        episodes=cfg.n_eval_episodes,
                        horizon=cfg.eval_horizon,
                        k_samples=cfg.eval_k_samples,
                        video_dir=_closed_loop_video_dir,
                        video_prefix=f'{_mname}_task{_ti:02d}_closed_loop',
                        video_episode=(0 if _ti == 0 else -1),
                        video_fps=20,
                    )
                    _task_runs.append({'task_label': _task_label, 'metrics': _cl})
                    print(f'    success_rate = {_cl["success_rate_mean"]:.3f} ± {_cl["success_rate_std"]:.3f}'
                          f' | return = {_cl["return_mean"]:.2f} | steps = {_cl["steps_mean"]:.1f}')
                except Exception as _cl_err:
                    print(f'    [WARN] {_mname} failed on {_task_label}: {_cl_err}')

            if _task_runs:
                _succ = np.array([t['metrics']['success_rate_mean'] for t in _task_runs], dtype=np.float32)
                _ret = np.array([t['metrics']['return_mean'] for t in _task_runs], dtype=np.float32)
                _steps = np.array([t['metrics']['steps_mean'] for t in _task_runs], dtype=np.float32)
                _agg = {
                    'num_tasks': int(len(_task_runs)),
                    'task_results': _task_runs,
                    'success_rate_mean': float(_succ.mean()),
                    'success_rate_std': float(_succ.std()),
                    'return_mean': float(_ret.mean()),
                    'return_std': float(_ret.std()),
                    'steps_mean': float(_steps.mean()),
                    'steps_std': float(_steps.std()),
                }
                _closed_loop_results[_mname] = _agg
                print(f'  Aggregate {_mname}: success_rate={_agg["success_rate_mean"]:.3f} ± {_agg["success_rate_std"]:.3f}'
                      f' | return={_agg["return_mean"]:.2f} | tasks={_agg["num_tasks"]}')

                if wandb_run is not None:
                    wandb.log({
                        f'closed_loop/{_mname}/success_rate': _agg['success_rate_mean'],
                        f'closed_loop/{_mname}/return': _agg['return_mean'],
                        f'closed_loop/{_mname}/steps': _agg['steps_mean'],
                        f'closed_loop/{_mname}/num_tasks': _agg['num_tasks'],
                    })
    else:
        print('  [SKIP] No tasks discovered for closed-loop evaluation.')
else:
    print('\n[INFO] LIBERO simulator not installed — closed-loop eval skipped.')
    print('       Re-run the install cell; libero + robomimic + gymnasium must install cleanly.')

# ---- Assemble results (eval-only output path) ----
results = {
    'run_name': run_name,
    'source_run_name': cfg.source_run_name,
    'eval_mode': 'mujoco_only',
    'latency_ms': {'liquid': liq_latency, 'pi0': pi0_latency},
    'closed_loop': _closed_loop_results,
    'checkpoint_paths': {
        'latest': str(latest_ckpt),
        'best_liquid': str(best_liq_ckpt),
        'best_pi0': str(best_pi0_ckpt),
    },
}
with open(results_json, 'w') as f:
    json.dump(results, f, indent=2)

with open(history_json, 'w') as f:
    json.dump({'mode': 'eval_only', 'eval_mode': 'mujoco_only', 'source_run_name': cfg.source_run_name}, f, indent=2)

if wandb_run is not None:
    wandb.log({
        'latency/liquid_ms': liq_latency,
        'latency/pi0_ms': pi0_latency,
    })

    art = wandb.Artifact(name=f'{run_name}-eval-artifacts', type='experiment-artifacts')
    for fpath in [history_json, results_json]:
        if Path(fpath).exists():
            art.add_file(str(fpath))
    if '_closed_loop_video_dir' in globals() and Path(_closed_loop_video_dir).exists():
        for _vp in sorted(Path(_closed_loop_video_dir).glob('*.mp4')):
            art.add_file(str(_vp))
    wandb_run.log_artifact(art)
    print('W&B eval artifact uploaded.')

print('\nSaved eval artifacts:')
print('  history   :', history_json)
print('  results   :', results_json)
print('  ckpt src  :', latest_ckpt)
print('  best liq  :', best_liq_ckpt)
print('  best pi0  :', best_pi0_ckpt)
print('\nSAFE MODE: no training loop and no checkpoint writes.')

In [ ]:
# Closed-loop evaluation scaffold aligned with shared-backbone receding-horizon usage

@torch.no_grad()
def policy_chunk_actions(policy, batch, k_samples=16, chunk_len=8):
    """Return a [B, chunk_len, A] action chunk from one policy query.

    Strategy is read from cfg.closed_loop_action_selection:
      'mode'   – deterministic: pick the highest-logit mixture component at t=0,
                 then use that component's means for all chunk steps.  Coherent
                 single-trajectory rollout, no sampling noise.
      default  – sample K trajectories, pick the one closest to the MDN mean
                 (best-of-K proxy), return its first chunk_len steps.

    chunk_len must be <= pred_horizon.  Extra steps are discarded.
    """
    out = policy(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    logits = out['aux']['logits']   # [B, T, M]
    mu     = out['aux']['mu']       # [B, T, M, A]
    log_sigma = out['aux']['log_sigma']

    chunk_len = min(int(chunk_len), mu.shape[1])
    strategy  = str(getattr(cfg, 'closed_loop_action_selection', 'mode')).lower()

    if strategy == 'mode':
        # Argmax mixture component at t=0; use that component's means for all steps.
        comp0 = logits[:, 0, :].argmax(dim=-1)        # [B]
        bidx  = torch.arange(mu.shape[0], device=mu.device)
        # Advanced indexing: select comp0's means across all chunk steps
        chunk = mu[bidx.unsqueeze(1).expand(-1, chunk_len),
                   torch.arange(chunk_len, device=mu.device).unsqueeze(0).expand(mu.shape[0], -1),
                   comp0.unsqueeze(1).expand(-1, chunk_len)]   # [B, chunk_len, A]
        return chunk  # [B, chunk_len, A]

    # Best-of-K proxy: sample K trajectories, pick closest to MDN mean
    samples  = sample_mdn_actions(logits, mu, log_sigma, K=k_samples)  # [B,K,T,A]
    gt_proxy = mdn_mean_actions(logits, mu)                             # [B,T,A]
    errs     = ((samples - gt_proxy.unsqueeze(1)) ** 2).mean(dim=(-1, -2))  # [B,K]
    best     = errs.argmin(dim=1)                                            # [B]
    bidx     = torch.arange(samples.shape[0], device=samples.device)
    return samples[bidx, best, :chunk_len]   # [B, chunk_len, A]


@torch.no_grad()
def policy_first_action(policy, batch, k_samples=16):
    """Single-step wrapper kept for backward compatibility.  Returns [B, A]."""
    chunk = policy_chunk_actions(policy, batch, k_samples=k_samples, chunk_len=1)
    return chunk[:, 0, :]   # [B, A]


def _extract_success_from_info(info):
    """Search common success-key names in env info dict.  Returns (float, key_used)."""
    if not isinstance(info, dict):
        return 0.0, None
    direct_keys = ['success', 'is_success', 'task_success', 'goal_achieved',
                   'episode_success', 'done_success', 'completed']
    for k in direct_keys:
        if k in info:
            return float(bool(info[k])), k
    # Nested sub-dicts
    for sub in ['metrics', 'episode', 'eval', 'info']:
        if isinstance(info.get(sub), dict):
            for k in direct_keys:
                if k in info[sub]:
                    return float(bool(info[sub][k])), f'{sub}.{k}'
    return 0.0, None


def _extract_video_frame(obs):
    if obs is None:
        return None

    frame = obs.get('agentview_image') if isinstance(obs, dict) else None
    if frame is None and isinstance(obs, dict):
        image_like_keys = [k for k in obs.keys() if 'image' in k.lower() or 'rgb' in k.lower()]
        if image_like_keys:
            frame = obs[image_like_keys[0]]

    if frame is None:
        return None

    frame = np.asarray(frame)
    if frame.ndim != 3:
        return None

    if frame.shape[0] in (1, 3, 4) and frame.shape[-1] not in (1, 3, 4):
        frame = np.transpose(frame, (1, 2, 0))

    if frame.shape[-1] == 1:
        frame = np.repeat(frame, 3, axis=-1)
    elif frame.shape[-1] >= 3:
        frame = frame[..., :3]
    else:
        return None

    if frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)

    return frame


def save_rollout_video(frames, out_path, fps=20):
    if not frames:
        raise ValueError('No frames available to save.')

    try:
        import imageio.v2 as imageio
    except Exception:
        import imageio

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(str(out_path), fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return out_path


def _safe_env_reset(env):
    out = env.reset()
    if isinstance(out, tuple):
        if len(out) >= 1:
            obs = out[0]
            info = out[1] if len(out) >= 2 and isinstance(out[1], dict) else {}
            return obs, info
    return out, {}


def _safe_env_step(env, action):
    out = env.step(action)
    if isinstance(out, tuple):
        if len(out) == 5:
            obs, reward, terminated, truncated, info = out
            return obs, reward, bool(terminated), bool(truncated), info if isinstance(info, dict) else {}
        if len(out) == 4:
            obs, reward, done, info = out
            return obs, reward, bool(done), False, info if isinstance(info, dict) else {}
    raise RuntimeError(f'Unexpected env.step() return format: {type(out)}')


@torch.no_grad()
def closed_loop_rollout(policy, env, tokenizer_fn, horizon=300, k_samples=16,
                        video_path=None, video_fps=20):
    """Receding-horizon rollout with action chunking.

    The policy is queried once per chunk.  `chunk_exec_len` consecutive actions
    from the predicted chunk are executed in the environment before the next
    query.  This matches the training objective (predicting pred_horizon steps)
    and follows the convention of ACT / Diffusion Policy papers.

    chunk_exec_len is read from cfg (default 8).  Set to 1 for step-by-step.
    """
    chunk_exec_len = int(getattr(cfg, 'chunk_exec_len', 8))
    chunk_exec_len = max(1, min(chunk_exec_len, int(getattr(cfg, 'pred_horizon', 16))))

    obs, _ = _safe_env_reset(env)
    total_reward = 0.0
    steps = 0
    done = False
    info = {}
    frames = []
    success_flag = 0.0
    success_key_used = None

    first_frame = _extract_video_frame(obs)
    if first_frame is not None:
        frames.append(first_frame)

    while not done and steps < horizon:
        # ── Query policy once for a full chunk ─────────────────────────────
        batch = tokenizer_fn(obs)
        batch = to_device(batch, device)
        chunk = policy_chunk_actions(
            policy, batch, k_samples=k_samples, chunk_len=chunk_exec_len
        )  # [1, chunk_exec_len, A]

        # ── Execute each action in the chunk ───────────────────────────────
        for t in range(chunk_exec_len):
            if done or steps >= horizon:
                break
            a = chunk[0, t].detach().cpu().numpy()   # [A]
            obs, reward, terminated, truncated, info = _safe_env_step(env, a)

            frame = _extract_video_frame(obs)
            if frame is not None:
                frames.append(frame)

            total_reward += float(reward)
            done = bool(terminated or truncated)
            steps += 1

            # Accumulate success across episode steps
            s_val, s_key = _extract_success_from_info(info)
            if s_val > success_flag:
                success_flag = s_val
                success_key_used = s_key

    if success_key_used:
        print(f'  [rollout] success key used: {success_key_used!r} | success={success_flag:.1f}')
    else:
        print(f'  [rollout] no success key found in info. Available keys: {list(info.keys())}')

    saved_video = None
    if video_path is not None and frames:
        try:
            saved_video = str(save_rollout_video(frames, video_path, fps=video_fps))
        except Exception as e:
            print(f'[WARN] Video save failed: {e}')

    return {
        'return': float(total_reward),
        'steps': int(steps),
        'success': float(success_flag),
        'video_path': saved_video,
        'num_video_frames': int(len(frames)),
        'chunk_exec_len': int(chunk_exec_len),
    }


@torch.no_grad()
def evaluate_closed_loop(policy, make_env_fn, tokenizer_fn, episodes=20, horizon=300, k_samples=10, seed=0, video_dir=None, video_prefix='rollout', video_episode=0, video_fps=20):
    returns, successes, steps_list = [], [], []
    saved_videos = []
    for ep in range(episodes):
        env = make_env_fn(seed + ep)
        episode_video_path = None
        if video_dir is not None and ep == int(video_episode):
            episode_video_path = Path(video_dir) / f'{video_prefix}_ep{ep:02d}.mp4'
        out = closed_loop_rollout(
            policy, env, tokenizer_fn, horizon=horizon, k_samples=k_samples,
            video_path=episode_video_path, video_fps=video_fps,
        )
        env.close()
        returns.append(out['return'])
        successes.append(out['success'])
        steps_list.append(out['steps'])
        if out.get('video_path'):
            saved_videos.append(out['video_path'])

    success_mean = float(np.mean(successes)) if successes else 0.0
    success_std = float(np.std(successes)) if successes else 0.0
    return {
        'return_mean': float(np.mean(returns)) if returns else 0.0,
        'return_std': float(np.std(returns)) if returns else 0.0,
        'success_rate': success_mean,
        'success_rate_mean': success_mean,
        'success_rate_std': success_std,
        'steps_mean': float(np.mean(steps_list)) if steps_list else 0.0,
        'steps_std': float(np.std(steps_list)) if steps_list else 0.0,
        'episodes': int(episodes),
        'video_paths': saved_videos,
        'videos': saved_videos,
    }

print(f'Closed-loop scaffold ready.'
      f' chunk_exec_len={getattr(cfg, "chunk_exec_len", 8)}'
      f' | pred_horizon={getattr(cfg, "pred_horizon", 16)}'
      f' | action_selection={getattr(cfg, "closed_loop_action_selection", "mode")}')

Closed-loop scaffold ready. Plug in LIBERO env + tokenizer_fn to run rollouts.


In [ ]:
# Progress patch for closed-loop evaluation (episode-level progress + timing)
from tqdm.auto import tqdm
import time

@torch.no_grad()
def evaluate_closed_loop(policy, make_env_fn, tokenizer_fn, episodes=20, horizon=300, k_samples=10, seed=0, video_dir=None, video_prefix='rollout', video_episode=0, video_fps=20, show_progress=True):
    returns, successes, steps_list = [], [], []
    saved_videos = []

    ep_iter = range(episodes)
    if show_progress:
        ep_iter = tqdm(ep_iter, desc=f'{video_prefix}: episodes', leave=False)

    t_start_all = time.perf_counter()
    for ep in ep_iter:
        t_ep = time.perf_counter()
        env = make_env_fn(seed + ep)
        episode_video_path = None
        if video_dir is not None and ep == int(video_episode):
            episode_video_path = Path(video_dir) / f'{video_prefix}_ep{ep:02d}.mp4'

        out = closed_loop_rollout(
            policy, env, tokenizer_fn, horizon=horizon, k_samples=k_samples,
            video_path=episode_video_path, video_fps=video_fps,
        )
        env.close()

        returns.append(out['return'])
        successes.append(out['success'])
        steps_list.append(out['steps'])
        if out.get('video_path'):
            saved_videos.append(out['video_path'])

        ep_sec = time.perf_counter() - t_ep
        if show_progress and hasattr(ep_iter, 'set_postfix'):
            ep_iter.set_postfix(
                ep=f'{ep+1}/{episodes}',
                steps=int(out['steps']),
                succ=f"{float(out['success']):.2f}",
                sec=f"{ep_sec:.1f}"
            )
        elif (ep + 1) % max(1, episodes // 5) == 0:
            print(f'  progress: {ep+1}/{episodes} episodes | last_ep={ep_sec:.1f}s | steps={int(out["steps"])}')

    total_sec = time.perf_counter() - t_start_all
    success_mean = float(np.mean(successes)) if successes else 0.0
    success_std = float(np.std(successes)) if successes else 0.0
    print(
        f'Closed-loop done: episodes={episodes}, horizon={horizon}, '
        f'total_time={total_sec/60.0:.1f} min, avg_ep={total_sec/max(1, episodes):.1f}s'
    )

    return {
        'return_mean': float(np.mean(returns)) if returns else 0.0,
        'return_std': float(np.std(returns)) if returns else 0.0,
        'success_rate': success_mean,
        'success_rate_mean': success_mean,
        'success_rate_std': success_std,
        'steps_mean': float(np.mean(steps_list)) if steps_list else 0.0,
        'steps_std': float(np.std(steps_list)) if steps_list else 0.0,
        'episodes': int(episodes),
        'video_paths': saved_videos,
        'videos': saved_videos,
        'total_time_sec': float(total_sec),
    }

print('Progress patch active: evaluate_closed_loop now shows per-episode progress/timing.')

In [ ]:
# Closed-loop only rerun (MuJoCo-only path)
import time

_bddl_root = _get_libero_bddl_root()
if _bddl_root is None:
    raise RuntimeError('Could not resolve LIBERO bddl_files root.')

_suite_bddl_dir = _bddl_root / str(cfg.libero_suite)
_bddl_files = sorted(_suite_bddl_dir.glob('*.bddl')) if _suite_bddl_dir.exists() else sorted(_bddl_root.rglob('*.bddl'))
assert _bddl_files, f"No BDDL files found under {_bddl_root}"

_eval_task_name = _bddl_files[0].stem.replace('_', ' ')
_eval_bddl_file_name = str(_bddl_files[0])
_task_label = _eval_task_name

print('BDDL file:', _eval_bddl_file_name)
print('task_name:', _eval_task_name)
print(f'Planned eval: 2 models × {cfg.n_eval_episodes} episodes/model × horizon {cfg.eval_horizon} steps')
print('Tip: first episode can be slower due to simulator/assets initialization.')

_make_env = make_libero_env_fn(
    _eval_task_name,
    image_size=cfg.image_size,
    bddl_file_name=_eval_bddl_file_name,
)
_tok_fn = make_tokenizer_fn(cfg, clip_tokenizer=globals().get('clip_tokenizer', None), task_name=_task_label)

_video_dir = artifact_path / 'closed_loop_videos' if 'artifact_path' in globals() else (Path.cwd() / 'artifacts' / 'closed_loop_videos')
_video_dir.mkdir(parents=True, exist_ok=True)

_all_start = time.perf_counter()
for name, model, ckpt in [("liquid", liquid, best_liq_ckpt), ("pi0", pi0, best_pi0_ckpt)]:
    _model_start = time.perf_counter()

    if ckpt.exists():
        payload = torch.load(ckpt, map_location=device)
        model.load_state_dict(payload['model'])
        print(f'Loaded best checkpoint for {name}: {ckpt}')
    else:
        print(f'[WARN] Best checkpoint missing for {name}; using in-memory weights.')

    out = evaluate_closed_loop(
        model, _make_env, _tok_fn,
        episodes=cfg.n_eval_episodes,
        horizon=cfg.eval_horizon,
        k_samples=cfg.eval_k_samples,
        video_dir=_video_dir,
        video_prefix=f'{name}_closed_loop',
        video_episode=0,
        video_fps=20,
        show_progress=True,
    )

    _model_sec = time.perf_counter() - _model_start
    print(name, out)
    print(f'  elapsed for {name}: {_model_sec/60.0:.1f} min')

    if out.get('video_paths'):
        print(f"  saved video: {out['video_paths'][0]}")

        if 'wandb_run' in globals() and (wandb_run is not None):
            try:
                wandb.log({
                    f'closed_loop/{name}/video': wandb.Video(out['video_paths'][0], fps=20, format='mp4'),
                    f'closed_loop/{name}/success_rate': out.get('success_rate_mean', 0.0),
                    f'closed_loop/{name}/return': out.get('return_mean', 0.0),
                    f'closed_loop/{name}/steps': out.get('steps_mean', 0.0),
                })
                print(f"  logged to W&B: {out['video_paths'][0]}")
            except Exception as _wb_err:
                print(f"  [WARN] W&B video log failed for {name}: {_wb_err}")

if 'wandb_run' in globals() and (wandb_run is not None):
    try:
        _all_videos = sorted(_video_dir.glob('*.mp4'))
        if _all_videos:
            _video_art = wandb.Artifact(name=f'{cfg.eval_run_name}-closed-loop-videos', type='rollout-videos')
            for _vp in _all_videos:
                _video_art.add_file(str(_vp))
            wandb_run.log_artifact(_video_art)
            print(f'Uploaded {len(_all_videos)} rollout video(s) to W&B artifact.')
    except Exception as _wb_art_err:
        print(f'[WARN] W&B video artifact upload failed: {_wb_art_err}')

print(f'Total rerun elapsed: {(time.perf_counter() - _all_start)/60.0:.1f} min')

In [ ]:
# Final paper-ready closed-loop summary table
from pathlib import Path
import json

# Resolve results payload from memory or disk
_payload = globals().get('results', None)
if _payload is None:
    _results_path = globals().get('results_json', Path.cwd() / 'artifacts' / 'results.json')
    _results_path = Path(_results_path)
    if not _results_path.exists():
        raise FileNotFoundError(f"No in-memory results and file not found: {_results_path}")
    with _results_path.open('r') as f:
        _payload = json.load(f)

_closed = _payload.get('closed_loop', {})

print("\nClosed-loop LIBERO summary (aggregated across evaluated tasks)")
print(f"{'Model':<10} {'Success Mean':>14} {'Success Std':>12} {'Num Tasks':>10} {'Return Mean':>12}")
print("-" * 66)
for _name in ('liquid', 'pi0'):
    _m = _closed.get(_name, {})
    print(
        f"{_name:<10} "
        f"{float(_m.get('success_rate_mean', 0.0)):>14.4f} "
        f"{float(_m.get('success_rate_std', 0.0)):>12.4f} "
        f"{int(_m.get('num_tasks', 0)):>10d} "
        f"{float(_m.get('return_mean', 0.0)):>12.4f}"
    )

# Optional per-task preview (first 5 tasks)
for _name in ('liquid', 'pi0'):
    _tasks = (_closed.get(_name, {}) or {}).get('task_results', [])
    if not _tasks:
        continue
    print(f"\n{_name} task-level preview (first {min(5, len(_tasks))}):")
    print(f"{'Task':<42} {'Success':>8} {'Return':>10}")
    for _t in _tasks[:5]:
        _label = str(_t.get('task_label', 'unknown'))[:42]
        _metrics = _t.get('metrics', {})
        print(f"{_label:<42} {float(_metrics.get('success_rate_mean', 0.0)):>8.3f} {float(_metrics.get('return_mean', 0.0)):>10.3f}")